# Tree-DTW playground — the **networkx rebuild** (`tree_dtw_nx`), interactive

Interactive tests + visualization for the rebuilt matcher in `network_matching.tree_dtw_nx` (docs `docs/tree_dtw_nx.md`). A source **tree** `A` and target **network** `B` are plain `networkx.DiGraph`s; the matcher runs the six parts — candidates → emission `E` → forward `D` → backward `B` → extraction → validation.

**Why this notebook and not `tree_dtw_playground.ipynb`.** That one drives the *old* `tree_dtw.match_tree_to_bgraph`, whose **segment validation collapses the arc match to a point matching** and prints phantom `V1` crosses (docs §8.1/§8.7). The rebuild validates on the graph it matches on — `A`/`B` (point) or `L(A)`/`L(B)` (segment) — and never collapses. It also adds the §6b **cross-table reciprocity** check.

**The plots are [Plotly](https://plotly.com/python/) — interactive.** The scenarios have `B` sitting ≈0.5 m from `A`, so they overlap; the plots use a **correspondence view** — **source `A`** (black) at its real position, **target `B`** (grey) lifted above it, **orange** lines + diamonds joining each matched `A`→`B` point. **Hover** a node or a match diamond for its id / pair, **drag** to zoom, and use the **dropdown** to switch scenario and point/segment mode — all without re-running the cell.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))
import importlib
import numpy as np
import plotly.graph_objects as go
import network_matching.tree_dtw_nx as tnx
importlib.reload(tnx)

# built-in synthetic scenarios: source tree A, target network B (both nx.DiGraph, nodes carry x, y)
def cases():
    return {
        "chain":   (tnx.digraph({0:(0,0),1:(10,0),2:(20,0)}, [(0,1),(1,2)]),
                    tnx.digraph({"b0":(0,.5),"b1":(10,.5),"b2":(20,.5)}, [("b0","b1"),("b1","b2")])),
        "y_split": (tnx.digraph({0:(0,0),1:(10,0),2:(20,6),3:(20,-6)}, [(0,1),(1,2),(1,3)]),
                    tnx.digraph({"s":(0,.5),"j":(10,.5),"u":(20,6.5),"d":(20,-5.5)}, [("s","j"),("j","u"),("j","d")])),
        "merge":   (tnx.digraph({0:(0,6),1:(0,-6),2:(10,0),3:(20,0)}, [(0,2),(1,2),(2,3)]),
                    tnx.digraph({"a":(0,6.5),"b":(0,-5.5),"m":(10,.5),"o":(20,.5)}, [("a","m"),("b","m"),("m","o")])),
    }
def _xy(G, n): return (G.nodes[n]["x"], G.nodes[n]["y"])
def run_point(A, B, r=20.0, alpha=1.0, beta=1.0):
    tnx.prepare(A, B, r=r); tnx.forward(A, B, alpha=alpha, beta=beta); tnx.backward(A, B, alpha=alpha, beta=beta)
    return tnx.extract(A, B)

# ---- Plotly correspondence view -------------------------------------------------------------------
_COL = dict(B="#9aa0a6", Bn="#5f6368", A="#111111", M="#ff7f0e")

def _dy(bgA, bgB):
    allx = [_xy(g,n)[0] for g in (bgA,bgB) for n in g.nodes]
    ally = [_xy(g,n)[1] for g in (bgA,bgB) for n in g.nodes]
    return (max(ally)-min(ally)) + 0.35*(max(allx)-min(allx)) + 3.0     # push target B above source A

def _combo(bgA, bgB, recs, vis):
    """Six traces for one (source, target, matching): B edges/nodes (lifted), A edges/nodes, match
    lines, match diamonds. `recs` = list of (a_id, b_id, ax, ay, bx, by)."""
    dy = _dy(bgA, bgB); T = []
    ex,ey=[],[]
    for u,v in bgB.edges: (x0,y0),(x1,y1)=_xy(bgB,u),_xy(bgB,v); ex+=[x0,x1,None]; ey+=[y0+dy,y1+dy,None]
    T.append(go.Scatter(x=ex,y=ey,mode="lines",line=dict(color=_COL["B"],width=7),hoverinfo="skip",visible=vis,showlegend=False))
    T.append(go.Scatter(x=[_xy(bgB,n)[0] for n in bgB.nodes],y=[_xy(bgB,n)[1]+dy for n in bgB.nodes],mode="markers",
                        marker=dict(color=_COL["Bn"],size=10),text=[f"B \u00b7 {n}" for n in bgB.nodes],
                        hovertemplate="target %{text}<extra></extra>",visible=vis,showlegend=False))
    ex,ey=[],[]
    for u,v in bgA.edges: (x0,y0),(x1,y1)=_xy(bgA,u),_xy(bgA,v); ex+=[x0,x1,None]; ey+=[y0,y1,None]
    T.append(go.Scatter(x=ex,y=ey,mode="lines",line=dict(color=_COL["A"],width=3),hoverinfo="skip",visible=vis,showlegend=False))
    T.append(go.Scatter(x=[_xy(bgA,n)[0] for n in bgA.nodes],y=[_xy(bgA,n)[1] for n in bgA.nodes],mode="markers",
                        marker=dict(color=_COL["A"],size=10),text=[f"A \u00b7 {n}" for n in bgA.nodes],
                        hovertemplate="source %{text}<extra></extra>",visible=vis,showlegend=False))
    mx,my=[],[]
    for (aid,bid,ax,ay,bx,by) in recs: mx+=[ax,bx,None]; my+=[ay,by+dy,None]
    T.append(go.Scatter(x=mx,y=my,mode="lines",line=dict(color=_COL["M"],width=2.5),hoverinfo="skip",visible=vis,showlegend=False))
    T.append(go.Scatter(x=[(ax+bx)/2 for (aid,bid,ax,ay,bx,by) in recs],y=[(ay+by+dy)/2 for (aid,bid,ax,ay,bx,by) in recs],
                        mode="markers",marker=dict(color=_COL["M"],size=8,symbol="diamond"),
                        text=[f"{aid} \u2192 {bid}" for (aid,bid,ax,ay,bx,by) in recs],
                        hovertemplate="match %{text}<extra></extra>",visible=vis,showlegend=False))
    return T

def _layout(fig, title, **kw):
    fig.update_layout(title=dict(text=title), height=560, width=620, plot_bgcolor="white",
                      xaxis=dict(visible=False), yaxis=dict(visible=False, scaleanchor="x", scaleratio=1),
                      margin=dict(l=10,r=10,t=80,b=10), **kw)
    return fig

def point_recs(name, r=20.0):
    A,B = cases()[name]; M,c = run_point(A,B,r=r)
    return A, B, [(a, c[a], *_xy(A,a), *_xy(B,c[a])) for a in A.nodes]
def segment_recs(name, bw=2.0, r=20.0):
    A,B = cases()[name]; LA,LB = tnx.line_digraph(A), tnx.line_digraph(B)
    tnx.prepare(LA,LB,r=r,bearing_weight=bw); tnx.forward(LA,LB); tnx.backward(LA,LB); M,c = tnx.extract(LA,LB)
    return A, B, [(s, c[s], *_xy(LA,s), *_xy(LB,c[s])) for s in LA.nodes]     # background = point graphs

def explorer():
    """One interactive figure with a dropdown over the three scenarios x {point, segment}."""
    combos = [(n,m) for m in ("point","segment") for n in ("chain","y_split","merge")]
    traces=[]; groups=[]
    for name,mode in combos:
        A,B,recs = (point_recs if mode=="point" else segment_recs)(name)
        ts=_combo(A,B,recs,False); groups.append((len(traces),len(ts))); traces+=ts
    fig=go.Figure(traces); n=len(traces); default=combos.index(("y_split","point"))
    s,cn=groups[default]
    for k in range(s,s+cn): fig.data[k].visible=True
    def mask(gi):
        m=[False]*n; s,cn=groups[gi]
        for k in range(s,s+cn): m[k]=True
        return m
    buttons=[dict(label=f"{nm} \u00b7 {md}",method="update",
                  args=[{"visible":mask(i)},{"title.text":f"{nm} \u2014 {md} mode"}]) for i,(nm,md) in enumerate(combos)]
    _layout(fig, f"{combos[default][0]} \u2014 {combos[default][1]} mode",
            updatemenus=[dict(active=default,buttons=buttons,x=0,xanchor="left",y=1.16,yanchor="top",showactive=True)])
    return fig

def perturbed(name, shift=0.0, rotate=0.0):
    """Rigidly rotate the source tree about its centroid and shift it north -- for the live playground."""
    A,B = cases()[name]; th=np.radians(rotate)
    xs=[A.nodes[k]["x"] for k in A.nodes]; ys=[A.nodes[k]["y"] for k in A.nodes]
    cx,cy=float(np.mean(xs)),float(np.mean(ys)); nodes={}
    for k in A.nodes:
        x,y=A.nodes[k]["x"]-cx, A.nodes[k]["y"]-cy
        nodes[k]=(x*np.cos(th)-y*np.sin(th)+cx, x*np.sin(th)+y*np.cos(th)+cy+shift)
    return tnx.digraph(nodes, list(A.edges)), B

def playground(name="y_split", mode="point", shift=0.0, rotate=0.0):
    A,B = perturbed(name, shift, rotate)
    if mode=="point":
        M,c = run_point(A,B,r=40.0); recs=[(a,c[a],*_xy(A,a),*_xy(B,c[a])) for a in A.nodes]
    else:
        LA,LB=tnx.line_digraph(A),tnx.line_digraph(B)
        tnx.prepare(LA,LB,r=40.0,bearing_weight=2.0); tnx.forward(LA,LB); tnx.backward(LA,LB); M,c=tnx.extract(LA,LB)
        recs=[(s,c[s],*_xy(LA,s),*_xy(LB,c[s])) for s in LA.nodes]
    return _layout(go.Figure(_combo(A,B,recs,True)), f"{name} \u2014 {mode}  (shift={shift}, rotate={rotate}\u00b0)")

## 1. Interactive explorer

**Hover** for node ids and matched pairs, **drag** to zoom (double-click to reset), and use the **dropdown** to switch scenario × mode. Segment mode shows the arc→arc match (orange joins arc **midpoints**); it is validated on `L(A)`/`L(B)`, so no phantom crosses.

In [2]:
explorer()

## 2. Validity + cross-table agreement (all scenarios)

`check_rules` (V1–V4), `validate_tables` (per-cell `D`/`B`), and `check_reciprocity` (§6b) for every scenario and mode.

In [3]:
print(f"{'scenario':10} {'mode':8} {'valid(V1-4)':12} {'D/B cells':10} {'reciprocity':11}")
for mode in ("point","segment"):
    for name in ("chain","y_split","merge"):
        A,B = cases()[name]
        if mode=="point":
            M,c = run_point(A,B); src,tgt = A,B
        else:
            src,tgt = tnx.line_digraph(A), tnx.line_digraph(B)
            tnx.prepare(src,tgt,r=20.0,bearing_weight=2.0); tnx.forward(src,tgt); tnx.backward(src,tgt); M,c = tnx.extract(src,tgt)
        v1,v2,v3 = tnx.check_rules(M, src, tgt)
        v4 = [a for a in src.nodes if not any(x==a for (x,_w) in M)]
        valid = not (v1 or v2 or v3 or v4)
        nD,badD = tnx.validate_tables(src,tgt,"D"); nB,badB = tnx.validate_tables(src,tgt,"B")
        recip = "AGREE" if not tnx.check_reciprocity(src,c) else "FAIL"
        print(f"{name:10} {mode:8} {str(valid):12} {f'{nD}/{nB} ok' if not (badD or badB) else 'BAD':10} {recip:11}")

scenario   mode     valid(V1-4)  D/B cells  reciprocity
chain      point    True         7/7 ok     AGREE      
y_split    point    True         12/12 ok   AGREE      
merge      point    True         12/12 ok   AGREE      
chain      segment  True         4/4 ok     AGREE      
y_split    segment  True         9/9 ok     AGREE      
merge      segment  True         9/9 ok     AGREE      


## 3. Live playground — sliders  *(needs a running kernel)*

Rigidly **shift** / **rotate** the source tree and re-match live; switch scenario and mode. (Interactive only with a running kernel; a static view shows the initial state.)

In [4]:
from ipywidgets import interact, Dropdown, FloatSlider
interact(playground,
         name=Dropdown(options=["chain","y_split","merge"], value="y_split", description="scenario"),
         mode=Dropdown(options=["point","segment"], value="point", description="mode"),
         shift=FloatSlider(min=-6, max=6, step=0.5, value=0, description="shift"),
         rotate=FloatSlider(min=-30, max=30, step=1, value=0, description="rotate\u00b0"));

interactive(children=(Dropdown(description='scenario', index=1, options=('chain', 'y_split', 'merge'), value='…

## 4. Cross-table agreement — the §6b reciprocity check

`validate_tables` checks each `D`/`B` cell in isolation; `check_reciprocity` checks the two tables **agree** — every source edge the forward back-pointers thread, the backward ones thread back:

$$(p,\ \mathrm{tail}(p)) \in \mathrm{bpD}[c][\mathrm{head}(c)] \iff (c,\ \mathrm{head}(c)) \in \mathrm{bpB}[p][\mathrm{tail}(p)]$$

It **AGREE**s on the split; then one deliberately severed backward pointer is **caught**. (It holds only on the committed matching — off the optimum `D` and `B` optimise differently-pinned subproblems, docs §6b.)

In [5]:
A, B = cases()["y_split"]
M, committed = run_point(A, B)
print("reciprocity      :", "AGREE" if not tnx.check_reciprocity(A, committed) else "FAIL")

t = tnx._advance_anchor(A, 1, committed[1], "bpB")     # vertex 1 is the split; it feeds successors 2, 3
print(f"bpB[1][{t}] before:", A.nodes[1]["cand"][t]["bpB"])
A.nodes[1]["cand"][t]["bpB"] = [(s,w) for (s,w) in A.nodes[1]["cand"][t]["bpB"] if s != 2]
print(f"bpB[1][{t}] after :", A.nodes[1]["cand"][t]["bpB"], "  (severed the 1->2 continuation)")
print("reciprocity now  :", tnx.check_reciprocity(A, committed) or "AGREE")

reciprocity      : AGREE
bpB[1][j] before: [(2, 'u'), (3, 'd')]
bpB[1][j] after : [(3, 'd')]   (severed the 1->2 continuation)
reciprocity now  : [(1, 2, 'forward 1->2 unmirrored: (2,u) not in bpB[1][j]')]
